In [0]:
%sql
-- MERGE incremental sales records into Gold FactSales table

MERGE INTO retail_lakehouse.gold.fact_sales tgt

-- Source dataset prepared for merge operation
USING (
    
    SELECT
        TransactionID,
        CustomerSK,
        ProductSK,
        StoreSK,
        Quantity,
        Amount,
        TxnDate

    FROM (

        -- Join Silver sales data with dimension tables
        -- and generate surrogate keys for fact table loading

        SELECT
            s.TransactionID,

            -- Lookup active customer surrogate key
            c.CustomerSK,

            -- Lookup product surrogate key
            p.ProductSK,

            -- Lookup store surrogate key
            st.StoreSK,

            -- Sales quantity from source
            s.Quantity,

            -- Derive sales amount
            s.Quantity * p.UnitPrice AS Amount,

            -- Transaction date
            s.TxnDate,

            -- Deduplicate records based on TransactionID
            ROW_NUMBER() OVER (
                PARTITION BY s.TransactionID
                ORDER BY c.CustomerSK
            ) AS rn

        FROM retail_lakehouse.silver.sales s

        -- Join active customer records only (SCD Type 2)
        JOIN retail_lakehouse.gold.dim_customer c
            ON s.CustomerID = c.CustomerID
            AND c.IsActive = TRUE

        -- Join product dimension
        JOIN retail_lakehouse.gold.dim_product p
            ON s.ProductID = p.ProductID

        -- Join store dimension
        JOIN retail_lakehouse.gold.dim_store st
            ON s.StoreID = st.StoreID

    ) deduped

    -- Keep only one record per TransactionID
    WHERE rn = 1

) src

-- Match existing fact records using TransactionID
ON tgt.TransactionID = src.TransactionID

-- Update existing records if transaction already exists
WHEN MATCHED THEN
UPDATE SET

    tgt.Quantity = src.Quantity,
    tgt.Amount = src.Amount,
    tgt.TxnDate = src.TxnDate

-- Insert new records if transaction does not exist
WHEN NOT MATCHED THEN

INSERT
(
    SalesSK,
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    Amount,
    TxnDate
)

VALUES
(
    -- Generate surrogate key
    monotonically_increasing_id(),

    src.TransactionID,
    src.CustomerSK,
    src.ProductSK,
    src.StoreSK,
    src.Quantity,
    src.Amount,
    src.TxnDate
);